# Ingest pose data with Harp timing (`pirouette_data.ingestion`)

This notebook uses the `pirouette_data.ingestion` module to:

1. Load every DeepLabCut pose `.h5` file in a local directory.
2. Pull the matching camera frame-index `.csv` files from the AWS S3 open bucket
   (`s3://aind-open-data/...`), which carry the Harp timestamp (`Seconds`) per frame.
3. Concatenate everything into a single, time-ordered DataFrame with appended timing columns:
   - `harp_time` — raw Harp timestamp (s)
   - `time_since_start` — seconds since the start of the experiment (first frame of the earliest CSV)
   - `datetime_pacific` — timezone-aware Pacific wall-clock datetime

The camera used to locate the CSVs on S3 is derived automatically from each pose filename
(e.g. `TopCamera_2026-06-11T03-00-00.h5` -> camera `TopCamera`).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from pirouette_data import ingestion

pd.set_option("display.max_columns", 40)

## Configuration

In [ ]:
# Local directory holding the DeepLabCut pose .h5 files
POSE_DIR = r"C:/Users/brandon.pratt/Desktop/data/body-kinematics/pose_data"

# S3 URI of the session's behavior-videos directory (parent of the per-camera folders).
# The camera sub-folder (e.g. TopCamera/) is resolved from each pose filename.
S3_VIDEO_URI = "s3://aind-open-data/854393_2026-06-09_19-34-26/behavior-videos"

## Inspect the pieces

The individual helpers are useful on their own for debugging.

In [ ]:
# Parse camera + timestamp from a filename
camera, timestamp, dt = ingestion.parse_camera_and_timestamp(
    "TopCamera_2026-06-11T03-00-00.h5"
)
print(f"camera={camera!r}  timestamp={timestamp!r}  dt={dt}")

# The Harp time (s) of the very first frame of the experiment
start_harp = ingestion.get_experiment_start_harp(S3_VIDEO_URI, camera)
print(f"experiment start (Harp s): {start_harp}")
print(f"experiment start (Pacific): {ingestion.harp_to_datetime([start_harp]).iloc[0]}")

## Build the combined dataset

This downloads the matching CSVs from S3 and loads the local `.h5` files, so the first run
takes a little while (a few tens of MB per CSV).

In [ ]:
df = ingestion.build_dataset(POSE_DIR, S3_VIDEO_URI)
print(f"shape: {df.shape}")
df.head()

In [ ]:
# The appended timing columns
df[["source_file", "frame", "harp_time", "time_since_start", "datetime_pacific"]].head()

In [ ]:
# Per-file summary: frame counts and time span
summary = (
    df.groupby("source_file")
    .agg(
        n_frames=("frame", "size"),
        start_pacific=("datetime_pacific", "min"),
        end_pacific=("datetime_pacific", "max"),
        start_since_start_s=("time_since_start", "min"),
        end_since_start_s=("time_since_start", "max"),
    )
    .reset_index()
)
summary

## Sanity checks

In [ ]:
print("Harp time monotonically increasing:", df["harp_time"].is_monotonic_increasing)
print("datetime_pacific tz:", df["datetime_pacific"].dt.tz)
print("total duration (h):", df["time_since_start"].iloc[-1] / 3600)

# Median inter-frame interval -> effective frame rate
dt_s = df["harp_time"].diff().median()
print(f"median inter-frame interval: {dt_s * 1e3:.3f} ms  (~{1 / dt_s:.1f} fps)")

## Quick plot: a body part over experiment time

In [ ]:
# Pick a tracked body part present in the flattened columns
bodyparts = sorted(
    {c.rsplit("_", 1)[0] for c in df.columns if c.endswith(("_x", "_y", "_likelihood"))}
)
print("body parts:", bodyparts)
bp = bodyparts[-1]

# Downsample for a fast overview plot
step = max(1, len(df) // 20000)
sub = df.iloc[::step]

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sub["time_since_start"] / 3600, sub[f"{bp}_x"], lw=0.5, label=f"{bp}_x")
ax.plot(sub["time_since_start"] / 3600, sub[f"{bp}_y"], lw=0.5, label=f"{bp}_y")
ax.set(xlabel="time since experiment start (h)", ylabel="pixel", title=f"{bp} position over time")
ax.legend()
plt.tight_layout()
plt.show()